In [1]:
%load_ext autoreload
%autoreload 2

from IPython.display import display, HTML

display(HTML("<style>.container { width:100% !important; }</style>"))

In [2]:
from ikflow.model_loading import get_ik_solver
from ikflow.ikflow_solver import draw_latent
from ikflow.config import DEVICE
import torch
import numpy as np
from time import time, sleep
import sys
import os
sys.path.append(os.path.abspath(os.path.join(os.curdir, '..')))
from src.utils import RepoDir, BuildEnv, DrawAxes
import numpy as np
from pydrake.all import (
    StartMeshcat, 
    RigidTransform,
    Quaternion,
)


model_name = "panda__full__lp191_5.25m"

ik_solver, hyper_parameters = get_ik_solver(model_name)
robot = ik_solver.robot


meshcat = StartMeshcat()
diagram = BuildEnv(meshcat=meshcat, directives_file = os.path.join(RepoDir(), "models/panda/panda_collision.yaml"))
plant = diagram.GetSubsystemByName("plant")
diagram_context = diagram.CreateDefaultContext()
plant_context = plant.GetMyContextFromRoot(diagram_context)
diagram.ForcedPublish(diagram_context)
frame = plant.GetBodyByName("panda_hand").body_frame()

ikflow/config.py | Using device: 'cuda:0'
WorldModel::LoadRobot: /home/tangles/.cache/jrl/urdfs/panda_arm_hand_formatted_link_filepaths_absolute.urdf
joint mimic: no multiplier, using default value of 1 
joint mimic: no offset, using default value of 0 
URDFParser: Link size: 17
URDFParser: Joint size: 12
Geometry: Loading 12 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link0.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link3.dae into Group
Geometry: Loading 4 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link4.dae into Group
Geometry: Loading 3 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes/visual/link5.dae into Group
Geometry: Loading 17 meshes from /home/tangles/Urop/ikflow/.venv/lib/python3.10/site-packages/jrl/urdfs/panda/meshes

INFO:drake:Meshcat listening for connections at http://localhost:7000


In [3]:
q = np.array([0.5, -1.2, 0.8, -2.1, 1.5, 1.8, 0.2, 0, 0])
plant.SetPositions(plant_context, q)
quat = frame.CalcPoseInWorld(plant_context).rotation().ToQuaternion().wxyz()
translation = frame.CalcPoseInWorld(plant_context).translation()
c = np.hstack([translation, quat])
c_tensor = torch.tensor(c, dtype=torch.float32, device=DEVICE).unsqueeze(0)  # Shape: [1, 8]
print(c_tensor)
c_torch = torch.cat([c_tensor, torch.zeros((1, 1), dtype=torch.float32, device=DEVICE)], dim=1)
q_batch = torch.tensor(q[:7], dtype=torch.float32, device=DEVICE).unsqueeze(0)

output, _ = ik_solver.nn_model(q_batch, c=c_torch, rev=False) ## ENCODER

print(output)

print(ik_solver.nn_model(output, c=c_torch, rev=True)) ## DECODER


# q = torch.from_numpy(q).float().to(DEVICE)
# q_batch = q.unsqueeze(0)  # Shape: [1, 7]

# latent, _ = ik_solver.nn_model(q_batch, c=c_torch, rev=False)


tensor([[-0.2227,  0.3926,  0.6805,  0.4118, -0.5296, -0.7235, -0.1628]],
       device='cuda:0')
tensor([[-1.8599,  0.1243, -1.0313, -0.6565, -1.0012, -2.2010, -1.0576]],
       device='cuda:0', grad_fn=<CatBackward0>)
(tensor([[ 0.5000, -1.2000,  0.8000, -2.1000,  1.5000,  1.8000,  0.2000]],
       device='cuda:0', grad_fn=<MmBackward0>), tensor([-34.6708], device='cuda:0', grad_fn=<AddBackward0>))


In [4]:
latent = torch.tensor(np.random.randn(1,ik_solver.network_width), device=DEVICE, dtype=torch.float32) 
T = RigidTransform(np.array([[0, 0, 1, -0.1],[1, 0, 0, 0],[0, 1, 0, 0],[0, 0, 0, 1]]))

def get_ik_solution(target_pose, latent = None):
    '''
    Computes IK solution using IKFlow, and visualizes target pose in MeshCat in Drake
        Target Pose is numpy array of shape (7,) in the format [x y z qw qx qy qz]'''
    target_pose = torch.tensor(
        target_pose, device=DEVICE
    )
    conditional = torch.cat([target_pose.expand((1, ik_solver.network_width)), torch.zeros((1, 1), dtype=torch.float32, device=DEVICE)], dim=1)
    sol = ik_solver.nn_model(latent, c=conditional, rev=True)
    q = np.zeros(plant.num_positions())
    q[:7] = sol[0].detach().cpu().numpy()
    plant.SetPositions(plant_context, q)
    diagram.ForcedPublish(diagram_context)

    pose = target_pose.detach().cpu().numpy()
    pose = RigidTransform(Quaternion(pose[3], pose[4], pose[5], pose[6]), [pose[0], pose[1], pose[2]])
    T_gripper = pose.multiply(T.inverse())

    DrawAxes(T_gripper, meshcat)
    return q

In [5]:
### QUESTION: Is the model roughly continuous in physical space with the same latent vector?
latent = torch.tensor(np.random.randn(1,ik_solver.network_width), device=DEVICE, dtype=torch.float32) 
start = np.array([0.5, 0.5, 0.7, 0.7071, -0.7071, 0, 0], dtype=np.float32)
end = np.array([-0.5, 0.5, 0.7, -0.6, 0.48, 0.64, 0], dtype=np.float32)

# Linear interpolation for position (first 3)
# SLERP for quaternion (last 4)
t = np.linspace(0, 1, 15, dtype=np.float32)

pos_start, quat_start = start[:3], start[3:]
pos_end, quat_end = end[:3], -end[3:]

positions = np.outer(1-t, pos_start) + np.outer(t, pos_end)

angle = np.arccos(np.clip(np.dot(quat_start, quat_end), -1, 1))
quats = np.array([
    (np.sin((1-ti)*angle) * quat_start + np.sin(ti*angle) * quat_end) / np.sin(angle)
    for ti in t
])

samples = np.hstack([positions, quats])


for sample in samples[15:22]:
    q = get_ik_solution(target_pose=sample, latent=latent)
    print(q)
    sleep(0.01)



In [6]:
meshcat2 = StartMeshcat()
diagram2 = BuildEnv(meshcat=meshcat2, directives_file = os.path.join(RepoDir(), "models/panda/panda_collision_copy.yaml"))
plant2 = diagram2.GetSubsystemByName("plant")
diagram_context2 = diagram2.CreateDefaultContext()
plant_context2 = plant2.GetMyContextFromRoot(diagram_context2)
diagram2.ForcedPublish(diagram_context2)


INFO:drake:Meshcat listening for connections at http://localhost:7001
==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html

==== LCM Warning ===
LCM detected that large packets are being received, but the kernel UDP
receive buffer is very small.  The possibility of dropping packets due to
insufficient buffer space is very high.

For more information, visit:
   https://lcm-proj.github.io/lcm/content/multicast-setup.html



In [ ]:
chosen_samples = [samples[0], samples[20], samples[40], samples[60], samples[80], samples[100], samples[120]]

qs = np.full(63, 0.04)
i = 0
for sample in chosen_samples:
    print(sample)
    q = get_ik_solution(target_pose=target_pose, latent=torch.tensor(sample.reshape(1,7), device=DEVICE, dtype=torch.float32))
    qs[i: i + 7] = q[:7]
    i += 9
    sleep(0.01)

opacity = 0.7
meshcat2.SetProperty(f"/drake/illustration/panda", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/panda2", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/panda3", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/panda4", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/panda5", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/panda6", "opacity", opacity)
meshcat2.SetProperty(f"/drake/illustration/panda7", "opacity", opacity)

plant2.SetPositions(plant_context2, qs)
diagram2.ForcedPublish(diagram_context2)

[0. 0. 0. 0. 0. 0. 0.]
[-0.13888429  0.5730307   0.09356646  0.26262915 -0.38358095  0.43640262
 -0.06417121]
[ 0.3688083   0.6813953   0.7397595   0.39360636 -0.10369273  0.2501405
  0.26951352]
[ 0.27869514  0.14855857  0.74627066 -0.21929729  0.497253    0.23298964
  0.83437   ]
[-0.04131461  0.4846017   0.35545632 -0.39042762  0.8515943   0.27366412
  0.35799092]
[ 0.09089555  0.06920704  0.14330466 -0.21963485  0.6232102  -0.15872636
  0.1995926 ]
[-0.16998847 -0.5400434  -0.53052896 -0.77378774  0.8053386  -0.17688747
  0.56662476]


In [7]:
get_ik_solution(target_pose = start, latent=latent)
frame = plant.GetBodyByName("panda_link8").body_frame()

pose = frame.CalcPoseInWorld(plant_context)
print(pose.translation())
print(pose.rotation().ToQuaternion().wxyz())


print(plant.GetPositionLowerLimits())
print(plant.GetPositionUpperLimits())



[0.50031702 0.49919965 0.69959309]
[ 0.65174677 -0.65672885  0.27122889  0.26527015]
[-2.8973 -1.7628 -2.8973 -3.0718 -2.8973 -0.0175 -2.8973  0.      0.    ]
[ 2.8973  1.7628  2.8973 -0.0698  2.8973  3.7525  2.8973  0.04    0.04  ]


In [8]:
### QUESTION 2: Is the model roughly continuous in latent space for the same target pose?
target_pose = np.array([0.5, 0.5, 0.7, 0.7071, -0.7071, 0, 0], dtype=np.float32)

def hit_and_run_ball_small_steps(n_samples, dim=7, radius=1.5, step_size=0.1, x0=None):
    if x0 is None:
        x0 = np.zeros(dim, dtype=np.float32)
    
    samples = [x0]
    x = x0.copy()
    
    while len(samples) < n_samples:
        # Random direction
        direction = np.random.randn(dim).astype(np.float32)
        direction /= np.linalg.norm(direction)
        
        # Find max step along direction
        a = np.float32(1.0)
        b = np.float32(2.0) * np.dot(x, direction)
        c = np.dot(x, x) - np.float32(radius**2)
        discriminant = b**2 - np.float32(4.0)*a*c
        t_max = (-b + np.sqrt(discriminant)) / (np.float32(2.0)*a)
        t_min = (-b - np.sqrt(discriminant)) / (np.float32(2.0)*a)
        
        # Sample point on line
        t_end = np.float32(np.random.uniform(t_min, t_max))
        
        # Take small steps along line
        n_steps = int(abs(t_end) / step_size) + 1
        for t in np.linspace(0, t_end, n_steps, dtype=np.float32):
            x_new = x + t * direction
            samples.append(x_new.copy())
            if len(samples) >= n_samples:
                break
        
        x = samples[-1]
    
    return np.array(samples[:n_samples], dtype=np.float32)

# Usage
samples = hit_and_run_ball_small_steps(1000, dim=7, step_size=0.1)


In [9]:
errors = []
for sample in samples:
    get_ik_solution(target_pose=target_pose, latent=torch.tensor(sample.reshape(1,7), device=DEVICE, dtype=torch.float32))
    real_pose = frame.CalcPoseInWorld(plant_context)

    errors.append(np.linalg.norm(real_pose.translation() - target_pose[:3]))
    
    sleep(0.01)



KeyboardInterrupt: 

In [ ]:
import matplotlib.pyplot as plt

plt.plot(errors)
plt.xlabel('Index')
plt.ylabel('Error')
plt.title('Error in position of NN over different z in random walk')
plt.grid(True)
plt.show()
